# 🛡️ Robustness & Safety: OOD Detection + Hallucination Guard

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mattral/production-vlm-engineering/blob/main/notebooks/colab/03_robustness_guard_colab.ipynb)

Part of **[production-vlm-engineering](https://github.com/Mattral/production-vlm-engineering)**.

Three production failure modes and their mitigations — all runnable on CPU:

1. **Natural perturbation robustness** — how a chart reader degrades under ImageNet-C corruptions  
2. **Per-sample OOD detection** — calibrated kNN embedding-space guard  
3. **Hallucination guard** — faithfulness-based pass/flag/reject decision

**No GPU required. Runtime: ~3 minutes.**

In [1]:
import subprocess, sys
result = subprocess.run(
    ["pip", "install", "git+https://github.com/Mattral/production-vlm-engineering.git",
     "--quiet", "--no-deps"],
    capture_output=True, text=True
)
if result.returncode != 0:
    subprocess.run(["pip","install","numpy","scipy","matplotlib","Pillow","pyyaml","--quiet"], check=True)
    subprocess.run(["git","clone","--depth=1",
                    "https://github.com/Mattral/production-vlm-engineering.git",
                    "/content/pvlm"], capture_output=True)
    sys.path.insert(0, "/content/pvlm/src")
    sys.path.insert(0, "/content/pvlm")
print("✅ Setup complete")

✅ Setup complete


In [2]:
import numpy as np
from production_vlm.robustness import (
    NaturalPerturbation, apply_perturbation,
    KNNOODDetector, HallucinationGuard, GuardConfig, GuardDecision,
)
from production_vlm.robustness.chart_reader import read_tallest_bar
from production_vlm.utils.synthetic_charts import generate_synthetic_chart
from production_vlm.utils.vision_encoder import SyntheticEmbeddingProxy
print("✅ Imports OK")
print(f"Available perturbations: {sorted(NaturalPerturbation.ALL.keys())}")

✅ Imports OK
Available perturbations: ['brightness', 'contrast', 'gaussian_blur', 'gaussian_noise', 'occlusion', 'rotation']


## Part 1 — Natural perturbation robustness sweep

We measure what fraction of bar charts a pixel-based reader correctly identifies under each perturbation type and severity level. This is the proxy task for asking: *"how brittle is the vision component to realistic input degradation?"*

In [3]:
charts = [generate_synthetic_chart(seed=i, chart_type="bar", render_image=True) for i in range(15)]

print(f"{'Perturbation':<18} {'sev=0.0':>7} {'sev=0.25':>8} {'sev=0.5':>8} {'sev=0.75':>9} {'sev=1.0':>8}")
print("-" * 62)
for kind in sorted(NaturalPerturbation.ALL.keys()):
    row = f"{kind:<18}"
    for sev in [0.0, 0.25, 0.5, 0.75, 1.0]:
        ok = sum(
            read_tallest_bar(
                apply_perturbation(c.image, kind, sev, seed=i).perturbed_image,
                len(c.categories), int(np.argmax(c.values)), plot_bbox=c.plot_bbox
            ).correct
            for i, c in enumerate(charts)
        )
        row += f"  {ok/15:>6.0%}"
    print(row)

Perturbation       sev=0.0 sev=0.25  sev=0.5  sev=0.75  sev=1.0
--------------------------------------------------------------
brightness          100%     100%     100%      100%     100%
contrast            100%     100%     100%      100%     100%
gaussian_blur       100%      33%      33%       33%      33%
gaussian_noise      100%     100%      13%       33%      33%
occlusion           100%     100%      93%       87%      87%
rotation            100%      40%      47%       40%      13%


**Key insight**: Brightness/contrast are fully robust because the reader uses **adaptive background detection** (samples actual image corners) rather than a fixed absolute threshold — a real design fix documented in the codebase. Blur/rotation are genuinely brittle at high severity — the correct, honest result.

## Part 2 — Per-sample OOD detection

`KNNOODDetector` calibrates its threshold from the reference set's own leave-one-out kNN similarity distribution — no fixed cosine cutoff.

**Empirically validated operating points:**

| `percentile` | FP rate | TP rate |
|---|---|---|
| 1 | 0.0% | 2.5% — near useless |
| **15** | **10-12%** | **97-100%** — good default |
| 20 | 17.5% | 100% — looser |

In [4]:
encoder = SyntheticEmbeddingProxy(embedding_dim=64, seed=0, shift_magnitude=12.0)

ref_charts = [generate_synthetic_chart(seed=i, render_image=False) for i in range(150)]
ref_emb = encoder.encode_charts(ref_charts, style_shift_flags=[False]*150)
detector = KNNOODDetector(ref_emb, k=5, percentile=15.0)
print(f"Calibrated threshold: {detector.threshold:.4f}")

# Score in-distribution holdout
holdout_emb = encoder.encode_charts(
    [generate_synthetic_chart(seed=2000+i, render_image=False) for i in range(40)],
    style_shift_flags=[False]*40
)
fp = sum(r.is_ood for r in detector.score_batch(holdout_emb)) / 40

# Score OOD (style-shifted) inputs
shifted_emb = encoder.encode_charts(
    [generate_synthetic_chart(seed=3000+i, style_shift=True, render_image=False) for i in range(40)],
    style_shift_flags=[True]*40
)
tp = sum(r.is_ood for r in detector.score_batch(shifted_emb)) / 40

print(f"In-distribution FP rate: {fp:.1%}  (calibrated at percentile=15)")
print(f"OOD (style-shifted) TP rate: {tp:.1%}")

Calibrated threshold: 0.8765
In-distribution FP rate: 10.0%
OOD (style-shifted) TP rate: 100.0%


## Part 3 — Hallucination guard

Three-tier decision (pass/flag/reject) rather than binary — because 'uncertain but plausible' and 'clearly wrong' warrant different responses.

In [5]:
guard = HallucinationGuard(GuardConfig(pass_threshold=0.6, flag_threshold=0.3))
evidence  = "South: 67.6 req/s; North: 50.0 req/s; East: 45.2 req/s"
reference = "South has throughput of 67.6 req/s, which is the highest."

cases = [
    ("✅ Correct answer",       "South shows throughput of 67.6 req/s, making it the highest."),
    ("⚠️  Plausible but vague", "Throughput values vary considerably across regions."),
    ("❌ Hallucinated number",  "South has throughput of 999.9 req/s, by far the highest."),
]

print(f"{'Case':<28} {'Decision':>10} {'Faithfulness':>14}")
print("-" * 55)
for label, pred in cases:
    r = guard.check(pred, reference, evidence)
    print(f"{label:<28} {r.decision.value:>10} {r.faithfulness.score:>14.3f}")
    if r.decision == GuardDecision.REJECT:
        print(f"  ↳ Safe fallback: '{r.output_text[:60]}...'")

Case                           Decision   Faithfulness
-------------------------------------------------------
✅ Correct answer            pass         0.800
⚠️  Plausible but vague       flag         0.320
❌ Hallucinated number      reject         0.000
  ↳ Safe fallback: 'I'm not confident enough in this answer to provide it...


## Production wrapper pattern

This is the full integration — OOD check on input, VLM call, faithfulness check on output:

```python
from production_vlm.robustness import KNNOODDetector, HallucinationGuard
from production_vlm.utils.vision_encoder import RealVisionEncoder

# One-time setup
encoder    = RealVisionEncoder("facebook/dinov2-base")
ood_guard  = KNNOODDetector(reference_embeddings, k=5, percentile=15.0)
hall_guard = HallucinationGuard()

def safe_infer(image, question, evidence_text):
    # 1. Input-layer: OOD check
    emb = encoder.encode([image])[0]
    ood = ood_guard.score(emb)
    if ood.is_ood:
        return {"status": "ood_flagged", "ood_score": ood.ood_score}
    
    # 2. VLM inference (your model here)
    raw_answer = vlm.generate(image=image, prompt=question)
    
    # 3. Output-layer: faithfulness check
    result = hall_guard.check(raw_answer, reference_answer, evidence_text)
    return {
        "status":       result.decision.value,   # "pass"/"flag"/"reject"
        "answer":       result.output_text,       # safe fallback if rejected
        "faithfulness": result.faithfulness.score,
    }
```

## 🎯 Try it yourself

Edit the `MY_PREDICTION` cell below:

In [6]:
MY_PREDICTION = "South region shows throughput of 67.6 req/s making it highest"
result = guard.check(MY_PREDICTION, reference, evidence)
print(f"Decision:     {result.decision.value}")
print(f"Faithfulness: {result.faithfulness.score:.3f}")
print(f"  Numeric:   {result.faithfulness.numeric.score:.3f}")
print(f"  Grounding: {result.faithfulness.grounding.score:.3f}")

---
**Back to**: [01 — Evaluation Metrics](01_evaluation_metrics_colab.ipynb) · [02 — Drift Detection](02_drift_detection_colab.ipynb)

**Repo**: [github.com/Mattral/production-vlm-engineering](https://github.com/Mattral/production-vlm-engineering)  
**References**: Hendrycks & Dietterich (2019) ImageNet-C · Madry et al. (2018) PGD · Es et al. (2023) RAGAS